# Boxed Rescue Notebook — Stage 2 Post-Processing

Generic post-processing notebook. Takes a previous experiment's `*_responses.jsonl`,
finds entries missing `\boxed{}` (the dominant failure mode), and re-prompts the
model to extract a final answer from the truncated reasoning.

## How it works

1. Read responses from a precomputed stage-1 experiment (loaded from a Kaggle dataset)
2. Identify rows where the scorer would mark `missing_boxed`
3. Re-prompt the model with: original question + truncated reasoning + "give the boxed answer"
4. Append the new boxed extraction onto the original response (don't replace)
5. Rebuild submission.csv

## Setup (one-time per experiment)

1. Upload the upstream experiment's `public_responses.jsonl` + `private_responses.jsonl` as a Kaggle dataset
2. In a new Kaggle notebook, attach: competition dataset, `151b-experiments` dataset, the responses dataset
3. Set `RESCUE_EXPERIMENT` in Cell 3 to the experiment slug (the rescue experiment's `config.json` says which upstream to rescue)
4. GPU: **T4 x2**
5. Save Version → Save & Run All (Commit)

Runtime: ~10 min model load + ~20-30 min for ~200 short generations = **~45 min**.


## 1. Install vLLM + libcuda symlink

Copy of `cse151b-notebook.ipynb` Cell 3. After running, **restart the session**
once before running the next cell.

In [ ]:
!pip install -q vllm transformers tqdm "antlr4-python3-runtime==4.11.1" "protobuf<6.0"

import subprocess, os

KNOWN_PATHS = [
    "/usr/local/nvidia/lib64/libcuda.so.1",
    "/usr/lib/x86_64-linux-gnu/libcuda.so.1",
    "/usr/lib/libcuda.so.1",
]
out = subprocess.run(["ldconfig", "-p"], capture_output=True, text=True).stdout
ldconfig_cands = [l.split("=>")[1].strip() for l in out.splitlines() if "libcuda.so.1" in l]
all_cands = KNOWN_PATHS + ldconfig_cands
libcuda = next((p for p in all_cands if os.path.exists(p) and "stubs" not in p), None) \
       or next((p for p in all_cands if os.path.exists(p)), None)

if not libcuda:
    result = subprocess.run(["find", "/usr", "-name", "libcuda.so.1", "-not", "-path", "*/stubs/*"],
                            capture_output=True, text=True)
    found = [p.strip() for p in result.stdout.splitlines() if p.strip()]
    libcuda = found[0] if found else None

assert libcuda, "libcuda.so.1 not found"
print(f"Real libcuda: {libcuda}")

stub_dir = "/kaggle/working/cuda_stubs"
os.makedirs(stub_dir, exist_ok=True)
stub = f"{stub_dir}/libcuda.so"
if os.path.lexists(stub): os.remove(stub)
os.symlink(libcuda, stub)
for var in ("LIBRARY_PATH", "LD_LIBRARY_PATH"):
    os.environ[var] = f"{stub_dir}:{os.environ.get(var, '')}"
subprocess.run(["rm", "-rf", "/root/.cache/flashinfer"], check=False)

print("Install complete. → Run → Restart session, then run the next cell.")


## 2. Load config — pick which experiment to rescue

Change `RESCUE_EXPERIMENT` to the rescue experiment slug. Its `config.json`
declares the upstream `source_experiment` and the Kaggle dataset slug that
holds the upstream responses.

In [ ]:
import json
import importlib.util
import os
import re
import sys
import glob
from pathlib import Path

# ─── Must be set BEFORE importing vllm ────────────────────────────────
os.environ["VLLM_USE_V1"]                    = "0"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
os.environ["PYTORCH_ALLOC_CONF"]             = "expandable_segments:True"

# Re-expose libcuda stub dir in case of kernel restart
_stub_dir = "/kaggle/working/cuda_stubs"
if os.path.isdir(_stub_dir):
    for _var in ("LIBRARY_PATH", "LD_LIBRARY_PATH"):
        os.environ[_var] = f"{_stub_dir}:{os.environ.get(_var, '')}"

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

# ── Experiment selection ─────────────────────────────────────────────
RESCUE_EXPERIMENT = "exp_012_boxed_rescue"

def _find_exp_file(exp_name, filename):
    candidates = [
        f"experiments/{exp_name}/{filename}",
        *glob.glob(f"/kaggle/input/**/experiments/{exp_name}/{filename}", recursive=True),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return None

_cfg_path = _find_exp_file(RESCUE_EXPERIMENT, "config.json")
assert _cfg_path, f"No config.json found for {RESCUE_EXPERIMENT!r}"
EXP_CONFIG = json.loads(open(_cfg_path).read())
print(f"Loaded config from {_cfg_path}")

SOURCE_EXP     = EXP_CONFIG["source_experiment"]
STAGE1_DATASET = EXP_CONFIG["stage1_dataset_name"]

# ── Locate stage-1 inputs ────────────────────────────────────────────
def _find_stage1(filename):
    # 1. Kaggle dataset by name
    for c in glob.glob(f"/kaggle/input/{STAGE1_DATASET}/{filename}") + \
             glob.glob(f"/kaggle/input/**/{STAGE1_DATASET}/{filename}", recursive=True):
        return c
    # 2. Local repo path (dev mode)
    local = _find_exp_file(SOURCE_EXP, filename)
    if local: return local
    return None

PUBLIC_RESP_INPUT  = _find_stage1("public_responses.jsonl")
PRIVATE_RESP_INPUT = _find_stage1("private_responses.jsonl")
print(f"Stage-1 public input:  {PUBLIC_RESP_INPUT}")
print(f"Stage-1 private input: {PRIVATE_RESP_INPUT}")
assert PUBLIC_RESP_INPUT and PRIVATE_RESP_INPUT, \
    f"Missing stage-1 inputs. Did you attach the '{STAGE1_DATASET}' Kaggle dataset?"

# ── Rescue prompts module ────────────────────────────────────────────
_prompts_path = _find_exp_file(RESCUE_EXPERIMENT, "prompts.py")
assert _prompts_path, f"No prompts.py for {RESCUE_EXPERIMENT}"
_spec = importlib.util.spec_from_file_location("rescue_prompts", _prompts_path)
_rp = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_rp)
RESCUE_SYSTEM_PROMPT_MATH = _rp.RESCUE_SYSTEM_PROMPT_MATH
RESCUE_SYSTEM_PROMPT_MCQ  = _rp.RESCUE_SYSTEM_PROMPT_MCQ
build_rescue_user_message = _rp.build_rescue_user_message
print(f"Loaded rescue prompts from {_prompts_path}")

# ── Output paths ─────────────────────────────────────────────────────
WORKING_DIR  = Path("/kaggle/working")
PUBLIC_OUT   = WORKING_DIR / "public_responses.jsonl"
PRIVATE_OUT  = WORKING_DIR / "private_responses.jsonl"
SUBMISSION   = WORKING_DIR / "submission.csv"
STATS_OUT    = WORKING_DIR / "rescue_stats.json"

# ── Inference config ─────────────────────────────────────────────────
RESCUE                 = EXP_CONFIG["rescue"]
MODEL_ID               = RESCUE["model_id"]
MAX_TOKENS             = RESCUE["max_tokens"]
TEMPERATURE            = RESCUE["temperature"]
TOP_P                  = RESCUE["top_p"]
TOP_K                  = RESCUE["top_k"]
MAX_INPUT_FROM_STAGE1  = RESCUE["max_input_tokens_from_stage1"]

VLLM                   = EXP_CONFIG.get("vllm", {})
VLLM_MAX_MODEL_LEN     = VLLM.get("max_model_len", 8192)
VLLM_MAX_NUM_SEQS      = VLLM.get("max_num_seqs", 32)
VLLM_BATCHED_TOKENS    = VLLM.get("max_num_batched_tokens", 16384)
VLLM_TP                = VLLM.get("tensor_parallel_size", 2)
VLLM_GPU_MEM_UTIL      = VLLM.get("gpu_memory_utilization", 0.90)
VLLM_ENFORCE_EAGER     = VLLM.get("enforce_eager", False)

# ── Competition data ─────────────────────────────────────────────────
COMP_DIR     = "/kaggle/input/competitions/cse-151-b-spring-2026-competition"
PUBLIC_DATA  = f"{COMP_DIR}/public.jsonl"
PRIVATE_DATA = f"{COMP_DIR}/private.jsonl"

print(f"Rescue experiment : {RESCUE_EXPERIMENT}")
print(f"Source experiment : {SOURCE_EXP}")
print(f"Rescue model      : {MODEL_ID}")
print(f"max_tokens={MAX_TOKENS}  T={TEMPERATURE}  max_model_len={VLLM_MAX_MODEL_LEN}  tp={VLLM_TP}")


## 3. Load rescue model

Defaults to base Qwen3-Thinking-2507 (generic — works regardless of which model
produced the upstream responses). Override `rescue.model_id` in config.json to
swap.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    dtype="float16",                       # T4 doesn't support bfloat16
    tensor_parallel_size=VLLM_TP,
    enforce_eager=VLLM_ENFORCE_EAGER,
    disable_custom_all_reduce=True,
    enable_prefix_caching=False,
    gpu_memory_utilization=VLLM_GPU_MEM_UTIL,
    max_model_len=VLLM_MAX_MODEL_LEN,
    trust_remote_code=True,
    max_num_seqs=VLLM_MAX_NUM_SEQS,
    max_num_batched_tokens=VLLM_BATCHED_TOKENS,
)

sampling_params = SamplingParams(
    n=1,
    max_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    min_p=0.0,
)
print("Model loaded.")


## 4. Identify rescue candidates

Matches `analyze.py`'s `missing_boxed` bucket: any response with no `\boxed`
substring anywhere. Conservative — we don't try to rescue responses where the
scorer already finds *some* boxed answer (even if wrong).

In [ ]:
def needs_rescue(response):
    """Mirror analyze.py: missing_boxed = no \\boxed substring anywhere."""
    return "\\boxed" not in (response or "")

def truncate_to_last_n_tokens(text, n_tokens, tok):
    """Keep only the last n_tokens of text (where the conclusion would live)."""
    ids = tok.encode(text, add_special_tokens=False)
    if len(ids) <= n_tokens:
        return text
    return tok.decode(ids[-n_tokens:], skip_special_tokens=True)

# Load stage-1 responses + competition questions
public_responses  = [json.loads(l) for l in open(PUBLIC_RESP_INPUT)]
private_responses = [json.loads(l) for l in open(PRIVATE_RESP_INPUT)]
public_qs  = {json.loads(l)["id"]: json.loads(l) for l in open(PUBLIC_DATA)}
private_qs = {json.loads(l)["id"]: json.loads(l) for l in open(PRIVATE_DATA)}

print(f"Loaded {len(public_responses)} public responses ({len(public_qs)} questions)")
print(f"Loaded {len(private_responses)} private responses ({len(private_qs)} questions)")

# Find candidates
public_rescue_items  = [r for r in public_responses  if needs_rescue(r["response"])]
private_rescue_items = [r for r in private_responses if needs_rescue(r["response"])]
print(f"\nPublic rescue candidates:  {len(public_rescue_items)}/{len(public_responses)} ({len(public_rescue_items)/len(public_responses):.1%})")
print(f"Private rescue candidates: {len(private_rescue_items)}/{len(private_responses)} ({len(private_rescue_items)/len(private_responses):.1%})")

def build_rescue_prompt(item, qs_lookup):
    q = qs_lookup[item["id"]]
    truncated = truncate_to_last_n_tokens(item["response"], MAX_INPUT_FROM_STAGE1, tokenizer)
    system = RESCUE_SYSTEM_PROMPT_MCQ if item.get("is_mcq") else RESCUE_SYSTEM_PROMPT_MATH
    user = build_rescue_user_message(q["question"], q.get("options"), truncated)
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True,
    )

public_prompts  = [build_rescue_prompt(r, public_qs)  for r in public_rescue_items]
private_prompts = [build_rescue_prompt(r, private_qs) for r in private_rescue_items]
print(f"\nBuilt {len(public_prompts)} public + {len(private_prompts)} private rescue prompts")

# Preview one
if public_prompts:
    print("\n── Preview public rescue prompt 0 (last 800 chars) ──")
    print(public_prompts[0][-800:])


## 5. Generate rescues + merge

vLLM batches the rescue prompts. Each generates up to `max_tokens` (default 2048).
After generation, we append successful rescues (those that produced a `\boxed{}`)
onto the original response. Failed rescues leave the original untouched.

In [ ]:
print(f"Generating rescues for {len(public_prompts)} public prompts...")
public_outs = llm.generate(public_prompts, sampling_params=sampling_params)
print(f"Generating rescues for {len(private_prompts)} private prompts...")
private_outs = llm.generate(private_prompts, sampling_params=sampling_params)


def merge_rescues(originals, rescue_items, rescue_outs):
    """Append rescue text to originals where the rescue produced \boxed{}."""
    rescue_by_id = {}
    for item, out in zip(rescue_items, rescue_outs):
        text = out.outputs[0].text.strip()
        if "\\boxed" in text:
            rescue_by_id[item["id"]] = text

    merged = []
    for r in originals:
        new_r = dict(r)
        if r["id"] in rescue_by_id:
            new_r["response"] = r["response"] + "\n\n[RESCUE EXTRACTION]:\n" + rescue_by_id[r["id"]]
            new_r["rescued"] = True
        else:
            new_r["rescued"] = False
        merged.append(new_r)
    return merged, len(rescue_by_id)


public_merged,  n_public_rescued  = merge_rescues(public_responses,  public_rescue_items,  public_outs)
private_merged, n_private_rescued = merge_rescues(private_responses, private_rescue_items, private_outs)

print(f"\nPublic  : rescued {n_public_rescued}/{len(public_rescue_items)} candidates")
print(f"Private : rescued {n_private_rescued}/{len(private_rescue_items)} candidates")

# Write merged responses
with open(PUBLIC_OUT,  "w") as f:
    for r in public_merged:  f.write(json.dumps(r) + "\n")
with open(PRIVATE_OUT, "w") as f:
    for r in private_merged: f.write(json.dumps(r) + "\n")

# Submission CSV (private only)
import csv
with open(SUBMISSION, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "response"])
    for r in private_merged:
        writer.writerow([r["id"], r["response"]])

# Stats
stats = {
    "rescue_experiment": RESCUE_EXPERIMENT,
    "source_experiment": SOURCE_EXP,
    "rescue_model": MODEL_ID,
    "public": {
        "total": len(public_responses),
        "candidates": len(public_rescue_items),
        "rescued": n_public_rescued,
        "rescue_rate_of_candidates": n_public_rescued / max(len(public_rescue_items), 1),
    },
    "private": {
        "total": len(private_responses),
        "candidates": len(private_rescue_items),
        "rescued": n_private_rescued,
        "rescue_rate_of_candidates": n_private_rescued / max(len(private_rescue_items), 1),
    },
}
with open(STATS_OUT, "w") as f:
    json.dump(stats, f, indent=2)

print("\n", json.dumps(stats, indent=2))
print(f"\nOutputs:")
print(f"  {PUBLIC_OUT}")
print(f"  {PRIVATE_OUT}")
print(f"  {SUBMISSION}")
print(f"  {STATS_OUT}")


## 6. Preview rescued responses

Quick visual check that the rescue extraction worked and the merged response
contains a `\boxed{}` at the end.

In [ ]:
rescued_sample = [r for r in public_merged if r.get("rescued")][:3]
for r in rescued_sample:
    print(f"\n── id={r['id']}  is_mcq={r.get('is_mcq')} ──")
    print("Last 600 chars of merged response:")
    print(r["response"][-600:])
    print("─" * 60)
